# Camada Gold

Execute as células em ordem, com um kernel novo do ambiente virtual do projeto. Este notebook lê a Silver, prepara as cinco tabelas analíticas e grava Parquets em `data/gold`.

O cadastro e as coordenadas contêm as 27 capitais. A população mantém sua data de referência original. As séries meteorológicas usam o calendário local de 2025, com uma linha por capital e dia, inclusive quando não há medição disponível.

Cada execução substitui a saída da fonte após validar a nova carga. A gravação usa pastas temporárias, trava por destino e recuperação de operações interrompidas. O pequeno arquivo `.write-lock` é reutilizado. Esta etapa prepara os dados para o carregamento posterior no PostgreSQL.

O acompanhamento usa somente o estado atual. Os valores permanecem nos arquivos das camadas; não são criados snapshots. Para atualizar os vínculos no PostgreSQL, execute a Gold e depois o Load.


## 1. Imports e localização do projeto e configurar o período

O ano seleciona o período das séries meteorológicas. A população conserva a data de referência da estimativa, independentemente desse ano.

A pasta Silver `inmet/boundary_2025` contém as primeiras horas UTC de 2026 necessárias para completar o último dia local de 2025. Essas horas não criam dias de análise em 2026. Execute a preparação dessa borda na Bronze e na Silver antes deste notebook.

In [ ]:
import os
import sys
from datetime import date
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


project_directory = Path.cwd().resolve()

# Procura a raiz sem depender de __file__, que não existe no notebook.
while (
    not (project_directory / "pipeline" / "gold").is_dir()
    or not (project_directory / "data").is_dir()
):

    if project_directory == project_directory.parent:
        raise FileNotFoundError(
            "Abra o notebook com o diretório de trabalho dentro do projeto Viajando_com_o_dado."
        )

    project_directory = project_directory.parent

# Usa os módulos da Gold explicitamente, sem confundir imports das outras camadas.
sys.path.insert(0, str(project_directory))

from pipeline.shared import execution
from pipeline.gold import geocoding
from pipeline.gold import ibge_municipalities
from pipeline.gold import ibge_population
from pipeline.gold import inmet
from pipeline.gold import open_meteo


DATA_DIRECTORY = project_directory / "data"
SILVER_DIRECTORY = DATA_DIRECTORY / "silver"
GOLD_DIRECTORY = DATA_DIRECTORY / "gold"
YEAR = "2025"
START_DATE = date.fromisoformat(f"{YEAR}-01-01")
END_DATE = date.fromisoformat(f"{YEAR}-12-31")
MAX_STATION_DISTANCE_KM = 50.0
MINIMUM_STATION_COVERAGE = 0.9

BOUNDARY_DIRECTORY = SILVER_DIRECTORY / "inmet" / f"boundary_{YEAR}"
local_filesystem_jar = (
    project_directory / "pipeline" / "bronze" / "lib" / "hadoop-bare-naked-local-fs-0.1.0.jar"
)

required_paths = [
    SILVER_DIRECTORY / "ibge" / "municipalities",
    SILVER_DIRECTORY / "ibge" / "population",
    SILVER_DIRECTORY / "open_meteo" / "geocoding",
    SILVER_DIRECTORY / "open_meteo" / "weather" / YEAR,
    SILVER_DIRECTORY / "inmet" / YEAR,
    BOUNDARY_DIRECTORY,
]

if os.name == "nt":
    required_paths.append(local_filesystem_jar)

for required_path in required_paths:

    if not required_path.exists():
        raise FileNotFoundError(f"Caminho necessário não encontrado: {required_path}")

print(f"Projeto: {project_directory}")
print(f"Entrada: {SILVER_DIRECTORY}")
print(f"Destino: {GOLD_DIRECTORY}")
print(f"Período meteorológico: {START_DATE} a {END_DATE}")


In [ ]:
session = execution.run_context(
    project_directory, parameters={"year": str(globals().get("YEAR", "2025"))}
)
state = session.__enter__()
print("Acompanhando a execução atual.")


## 2. Importar os módulos

As funções validam as entradas antes de publicar cada tabela. Reinicie o kernel se tiver alterado os módulos de `pipeline/shared`: `storage`, `quality` ou `capitals`. O INMET utiliza `timezonefinder`, além do PySpark instalado no ambiente.

## 3. Iniciar o Spark

A execução local usa dois núcleos. Os instantes são apresentados em UTC na sessão; as funções calculam os dias locais de cada capital. No Windows, o JAR da Bronze configura o acesso aos arquivos locais.

In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable

spark_builder = (
    SparkSession.builder.appName("Viajando com o dado - Gold")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.hadoop.mapreduce.fileoutputcommitter.marksuccessfuljobs", "false")
    .config("spark.pyspark.python", sys.executable)
)

if os.name == "nt":
    spark_builder = (
        spark_builder.config("spark.driver.extraClassPath", str(local_filesystem_jar))
        .config("spark.executor.extraClassPath", str(local_filesystem_jar))
        .config(
            "spark.hadoop.fs.file.impl", "com.globalmentor.apache.hadoop.fs.BareLocalFileSystem"
        )
    )

spark = spark_builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print(f"Versão do Spark: {spark.version}")


## 4. Municípios IBGE

Valida os IDs oficiais, nomes, UFs e divisões regionais das 27 capitais. Mantém as origens e identifica cada registro com `is_capital`.

In [ ]:
municipalities = ibge_municipalities.build_municipalities(
    spark,
    SILVER_DIRECTORY / "ibge" / "municipalities",
    GOLD_DIRECTORY / "ibge" / "municipalities",
)

municipalities.printSchema()
municipalities.select(
    "municipality_id",
    "name",
    "state_code",
    "region_name",
    "immediate_region_name",
    "is_capital",
).orderBy("state_code").show(5, truncate=False)


## 5. População IBGE

Integra as estimativas ao cadastro das capitais, exigindo uma estimativa válida para cada capital e data de referência presente na entrada. Preserva as notas da fonte e não soma estimativas de datas diferentes.

In [ ]:
population = ibge_population.build_population(
    spark,
    SILVER_DIRECTORY / "ibge" / "population",
    GOLD_DIRECTORY / "ibge" / "municipalities",
    GOLD_DIRECTORY / "ibge" / "population",
)

population.printSchema()
population.select(
    "municipality_id",
    "name",
    "state_code",
    "population_estimate",
    "reference_date",
    "population_note",
).orderBy("reference_date", "state_code").show(5, truncate=False)


## 6. Geocoding

Confere as coordenadas com o cadastro das 27 capitais e preserva a origem das duas fontes. Esta saída fornece as coordenadas para selecionar as estações INMET e compor o calendário meteorológico.

In [ ]:
coordinates = geocoding.build_geocoding(
    spark,
    SILVER_DIRECTORY / "open_meteo" / "geocoding",
    GOLD_DIRECTORY / "ibge" / "municipalities",
    GOLD_DIRECTORY / "open_meteo" / "geocoding",
)

coordinates.printSchema()
coordinates.select(
    "municipality_id",
    "name",
    "state_code",
    "latitude",
    "longitude",
).orderBy(
    "state_code"
).show(5, truncate=False)


## 7. Open-Meteo diário

Gera o calendário das capitais e confere a coerência entre valores e alertas da Silver. Uma variável inválida fica nula na Gold, preservando as demais variáveis válidas do mesmo dia. Os indicadores `*_available` e `record_status` diferenciam registros ausentes, incompletos e inválidos. A presença de um valor diário não comprova cobertura horária completa.

In [ ]:
weather = open_meteo.build_open_meteo(
    spark,
    SILVER_DIRECTORY / "open_meteo" / "weather" / YEAR,
    GOLD_DIRECTORY / "open_meteo" / "geocoding",
    GOLD_DIRECTORY / "open_meteo" / "weather" / YEAR,
    START_DATE,
    END_DATE,
)

weather.printSchema()
weather.select(
    "name",
    "state_code",
    "observation_date",
    "temperature_mean_c",
    "precipitation_sum_mm",
    "wind_speed_max_ms",
    "record_status",
    "invalid_fields",
).orderBy("state_code", "observation_date").show(5, truncate=False)


## 8. INMET diário

Escolhe uma estação fixa por capital, na mesma UF, a até 50 km e com pelo menos 90% de cobertura válida de temperatura e chuva no período. Entre as elegíveis, prefere a mais próxima. Uma capital sem estação elegível permanece no calendário com `assignment_status = no_eligible_station`.

A média diária de temperatura exige ao menos 90% das horas esperadas. Chuva acumulada, temperaturas mínima e máxima, vento máximo e rajada máxima exigem todas as horas esperadas. A duração do dia considera o fuso local; campos sem cobertura suficiente ficam nulos. As séries INMET e Open-Meteo permanecem separadas, sem preencher automaticamente as lacunas de uma com a outra.

Esta é a etapa mais demorada: lê a base horária, avalia as estações e agrega as medições por dia local. A borda é passada explicitamente e os metadados da estação são conferidos antes de utilizá-la.

In [ ]:
station_weather = inmet.build_inmet(
    spark,
    SILVER_DIRECTORY / "inmet" / YEAR,
    GOLD_DIRECTORY / "open_meteo" / "geocoding",
    GOLD_DIRECTORY / "inmet" / YEAR,
    START_DATE,
    END_DATE,
    max_distance_km=MAX_STATION_DISTANCE_KM,
    minimum_station_coverage=MINIMUM_STATION_COVERAGE,
    boundary_directory=BOUNDARY_DIRECTORY,
)

station_weather.printSchema()
station_weather.select(
    "name",
    "state_code",
    "observation_date",
    "station_code",
    "station_distance_km",
    "temperature_mean_c",
    "precipitation_sum_mm",
    "temperature_coverage",
    "precipitation_coverage",
    "assignment_status",
    "boundary_metadata_status",
).orderBy("state_code", "observation_date").show(5, truncate=False)


## 9. Conferir as saídas

Verifica a quantidade e a unicidade dos registros das cinco tabelas. As séries devem conter as 27 capitais em todos os dias do período; isso garante o calendário, não a disponibilidade de todas as medições. Os resumos abaixo mostram separadamente os estados dos registros Open-Meteo e as capitais sem estação INMET elegível.

In [ ]:
expected_daily_rows = 27 * ((END_DATE - START_DATE).days + 1)
expected_population_rows = 27 * population.select("reference_date").distinct().count()

table_checks = [
    ("Municípios IBGE", municipalities, ["municipality_id"], 27),
    ("População IBGE", population, ["municipality_id", "reference_date"], expected_population_rows),
    ("Geocoding", coordinates, ["municipality_id"], 27),
    ("Open-Meteo diário", weather, ["municipality_id", "observation_date"], expected_daily_rows),
    ("INMET diário", station_weather, ["municipality_id", "observation_date"], expected_daily_rows),
]

summary_records = []

for source_name, dataframe, key_columns, expected_rows in table_checks:

    record_count = dataframe.count()
    repeated_keys = dataframe.groupBy(*key_columns).count().filter(F.col("count") > 1)

    if record_count != expected_rows or repeated_keys.limit(1).count() > 0:
        raise ValueError(
            f"Quantidade ou unicidade inesperada em {source_name}: {record_count} registros; esperado: {expected_rows}."
        )

    summary_records.append((source_name, record_count))

spark.createDataFrame(summary_records, "source string, record_count long").show(truncate=False)
weather.groupBy("record_status").count().orderBy("record_status").show(truncate=False)

# Conta capitais, sem multiplicar o resultado pelo número de dias do calendário.
station_assignments = station_weather.select(
    "municipality_id",
    "name",
    "state_code",
    "station_code",
    "assignment_status",
).distinct()

station_assignments.groupBy("assignment_status").count().show(truncate=False)
station_assignments.filter(F.col("assignment_status") == "no_eligible_station").select(
    "name",
    "state_code",
).orderBy("state_code").show(27, truncate=False)

print(f"Saídas da Gold: {GOLD_DIRECTORY}")


## 10. Encerrar a sessão

Encerre o Spark ao terminar. Para executar o notebook novamente após essa célula, reinicie a partir da configuração da sessão.

In [ ]:
execution.finish_notebook(session, spark)
